# 5 — Shrimp impact
Does the behaviour of the **shrimp** influence the **strategy** chosen by the cuttlefish?

- shrimp metrics: `add_features_shrimp.py` → `Data/shrimp_features_smooth_1.csv`
  (30 s before the start of the cuttlefish activity → strike, column `phase` = `pre` / `hunt`)
- cuttlefish strategies: a session computed by `1_compute.ipynb` (`CUTTLE_SESSION`, e.g. `reference_best`)
- optional: shrimp behavioural groups = a session computed by `1_compute.ipynb` with `SUBJECT = "shrimp"`

Only the `pre` phase can show an **influence** of the shrimp (the cuttlefish is not hunting yet).
In the `hunt` phase the shrimp can also be **reacting** to the strategy.

In [1]:
# ── Repository folders (RELATIVE paths: this notebook is in Code/) ──
from pathlib import Path
REPO_DIR    = Path.cwd().parent if Path.cwd().name == "Code" else Path.cwd()
DATA_DIR    = REPO_DIR / "Data"
RESULTS_DIR = REPO_DIR / "Results"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

STRATEGY_PALETTE = ["#FF4444", "#44AAFF", "#44FF88", "#FFD700", "#FF44FF",
                    "#FF8844", "#44FFFF", "#FF44AA", "#88FF44", "#AA44FF"]
PHASE_COLOR = "#00E5FF"      # start of the cuttlefish activity (same colour as the other notebooks)

In [2]:
# ── Display mode ─────────────────────────────────────────────────
#   True  : each figure opens in its OWN WINDOW, one after the other —
#           close it and the next one opens (the notebook waits for you)
#   False : figures are shown inside the notebook (classic inline mode)
#   WAIT_FOR_CLOSE (windows only):
#     True  : the notebook waits until you close each window (use with "Run All")
#     False : each cell opens its window and returns at once -> several windows
#             open at the same time (run the cells one by one, not "Run All")
#   SAVE_FIGURES : every displayed figure is also saved as PNG in the folder of the
#                  LOADED session: Results/<LOAD_NAME>/figures_5_shrimp_impact/  (rerunning a cell overwrites its file)
WAIT_FOR_CLOSE = False
SHOW_IN_WINDOWS = True
SAVE_FIGURES = True
FIG_DPI = 150

import re, hashlib
import matplotlib.pyplot as plt
import matplotlib.figure as _mfig

# figures already saved explicitly (plt.savefig in the code) are not saved a second time
if not hasattr(_mfig.Figure, "_savefig_original"):
    _mfig.Figure._savefig_original = _mfig.Figure.savefig
    def _savefig_mark(self, *a, **k):
        self._cs_saved = True
        return _mfig.Figure._savefig_original(self, *a, **k)
    _mfig.Figure.savefig = _savefig_mark

_fig_names = {}                                   # file name -> cell that produced it


def _fig_title(fig):
    t = fig._suptitle.get_text() if getattr(fig, "_suptitle", None) else ""
    for ax in ([] if t else fig.axes):
        t = ax.get_title() or ax.get_title(loc="left")
        if t:
            break
    return t or "figure"


def _autosave_figures():
    if not SAVE_FIGURES or globals().get("LOAD_DIR") is None:
        return
    fig_dir = Path(LOAD_DIR) / "figures_5_shrimp_impact"
    fig_dir.mkdir(parents=True, exist_ok=True)
    try:
        cell = hashlib.md5(get_ipython().user_ns["_ih"][-1].encode()).hexdigest()
    except Exception:
        cell = ""
    for num in plt.get_fignums():
        fig = plt.figure(num)
        if getattr(fig, "_cs_saved", False):
            continue
        base = re.sub(r"[^\w-]+", "_", _fig_title(fig)).strip("_")[:70] or "figure"
        name, i = base, 2
        while _fig_names.get(name, cell) != cell:     # same title from ANOTHER cell -> suffix
            name, i = f"{base}_{i}", i + 1
        _fig_names[name] = cell
        try:
            fig.savefig(fig_dir / f"{name}.png", dpi=FIG_DPI, bbox_inches="tight")
            print(f"[SAVED] {fig_dir.parent.name}/{fig_dir.name}/{name}.png")
        except Exception as e:
            print(f"[SAVED] failed for '{name}': {e}")


if SHOW_IN_WINDOWS:
    get_ipython().run_line_magic("matplotlib", "qt")
    _show_qt = getattr(plt, "_show_qt_original", plt.show)
    plt._show_qt_original = _show_qt          # survives a rerun of this cell
    _waiting = {"on": False}

    def _show_and_wait(*args, **kwargs):
        # plt.pause() calls plt.show() internally: while waiting, just display
        if _waiting["on"]:
            return _show_qt(block=False)
        figs = plt.get_fignums()
        if not figs:
            return
        _autosave_figures()
        _show_qt(block=False)
        if not WAIT_FOR_CLOSE:
            return
        print(f"[FIGURE] {len(figs)} window(s) open — close to continue")
        _waiting["on"] = True
        try:
            while any(plt.fignum_exists(n) for n in figs):
                plt.pause(0.2)
        finally:
            _waiting["on"] = False

    plt.show = _show_and_wait
else:
    get_ipython().run_line_magic("matplotlib", "inline")
    _show_inline = getattr(plt, "_show_qt_original", plt.show)

    def _show_inline_save(*args, **kwargs):
        _autosave_figures()
        return _show_inline(*args, **kwargs)

    plt.show = _show_inline_save
print(f"[DISPLAY] {('one window after the other' if WAIT_FOR_CLOSE else 'windows, several at once') if SHOW_IN_WINDOWS else 'inline'}"
      f" | figures {'saved in Results/<LOAD_NAME>/figures_5_shrimp_impact/' if SAVE_FIGURES else 'not saved'}")

[DISPLAY] windows, several at once | figures saved in Results/<LOAD_NAME>/figures_5_shrimp_impact/


## Settings + loading

In [ ]:
CUTTLE_SESSION = "reference_best"   # cuttlefish model: its strategy groups (mapping_optimal.json)
SHRIMP_SESSION = "_test_shrimp"               # shrimp model (1_compute with SUBJECT = "shrimp"), e.g. "shrimp_v1"
                                    #   None = no shrimp groups (the metrics are still compared)
SHRIMP_CSV     = DATA_DIR / "shrimp_features_smooth_1.csv"
INCLUDE_MONO   = False              # include the "MONO" group of the cuttlefish
EXCLUDE_ANIMALS = [152, 133]        # same exclusions as the cuttlefish analysis

# figures of this notebook are saved in the cuttlefish session: Results/<CUTTLE_SESSION>/figures_5_shrimp_impact/
LOAD_DIR = RESULTS_DIR / CUTTLE_SESSION

# ── Cuttlefish strategies ──
mapping_cuttle = {int(k): v for k, v in json.load(open(LOAD_DIR / "mapping_optimal.json")).items()}
if not INCLUDE_MONO:
    mapping_cuttle = {k: v for k, v in mapping_cuttle.items() if v != "MONO"}
STRATS = sorted(set(mapping_cuttle.values()))
PAL = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)] for i, s in enumerate(STRATS)}
PAL["MONO"] = "#888888"

# ── Shrimp groups (optional) ──
mapping_shrimp = None
if SHRIMP_SESSION:
    _cfg = json.load(open(RESULTS_DIR / SHRIMP_SESSION / "config.json", encoding="utf-8"))
    if _cfg.get("SUBJECT") != "shrimp":
        print(f"  ⚠️  {SHRIMP_SESSION} is not a shrimp session (SUBJECT={_cfg.get('SUBJECT')})")
    mapping_shrimp = {int(k): v for k, v in json.load(open(RESULTS_DIR / SHRIMP_SESSION / "mapping_optimal.json")).items()}

# ── Shrimp metrics, frame by frame ──
_cols = ["animal_id", "frame", "phase", "time_from_activity_s", "pre_duration_s",
         "shrimp_speed_mm_s", "shrimp_accel_peak", "shrimp_swim_dir_deg", "shrimp_forward_speed",
         "shrimp_turn_speed_deg_s", "shrimp_view_cuttle", "distance_mm"]
sh = pd.read_csv(SHRIMP_CSV, usecols=_cols)
sh = sh[~sh["animal_id"].isin(EXCLUDE_ANIMALS)]
sh["strategy"] = sh["animal_id"].map(mapping_cuttle)
sh = sh.dropna(subset=["strategy"]).sort_values(["animal_id", "frame"]).reset_index(drop=True)
print(f"[OK] cuttlefish session {CUTTLE_SESSION}: strategies {STRATS}")
print(f"[OK] shrimp metrics: {len(sh)} frames | {sh['animal_id'].nunique()} individuals with a strategy")
print(sh.groupby("strategy")["animal_id"].nunique().rename("n individuals").to_frame().T.to_string())

[OK] cuttlefish session reference_best: strategies ['A', 'B', 'C', 'D', 'E', 'F']
[OK] shrimp metrics: 337092 frames | 173 individuals with a strategy
strategy        A   B   C   D   E   F
n individuals  44  32  30  21  23  23


## Per-individual summary of each phase

In [4]:
def _summary(g):
    dt = g["time_from_activity_s"].diff().median()
    dur_s = len(g) * (dt if np.isfinite(dt) and dt > 0 else 1 / 60)
    moving = g["shrimp_swim_dir_deg"].dropna()
    return pd.Series({
        "duration (s)":            dur_s,
        "speed (mm/s)":            g["shrimp_speed_mm_s"].mean(),
        "% still":                 (g["shrimp_speed_mm_s"] == 0).mean() * 100,
        "accel. peaks / min":      g["shrimp_accel_peak"].sum() / dur_s * 60,
        "% swimming backwards":    (moving > 90).mean() * 100 if len(moving) else np.nan,
        "turning (°/s)":           g["shrimp_turn_speed_deg_s"].mean(),
        "cuttlefish seen at (°)":  g["shrimp_view_cuttle"].median(),
        "distance (mm)":           g["distance_mm"].median(),
    })

summary = (sh.groupby(["animal_id", "phase"]).apply(_summary).reset_index())
summary["strategy"] = summary["animal_id"].map(mapping_cuttle)

# state of the shrimp AT the start of the cuttlefish activity (first "hunt" frame)
_at = sh[sh["phase"] == "hunt"].groupby("animal_id").first()
at_start = pd.DataFrame({"distance at activity start (mm)": _at["distance_mm"],
                         "cuttlefish seen at activity start (°)": _at["shrimp_view_cuttle"]})
at_start["strategy"] = at_start.index.map(mapping_cuttle)

METRICS = ["speed (mm/s)", "% still", "accel. peaks / min", "% swimming backwards",
           "turning (°/s)", "cuttlefish seen at (°)", "distance (mm)"]
n_pre = summary[(summary["phase"] == "pre") & (summary["duration (s)"] > 0)]["animal_id"].nunique()
print(f"[OK] {summary['animal_id'].nunique()} individuals | with a 'pre' phase: {n_pre}")
summary.groupby(["phase", "strategy"])[METRICS].median().round(1)

[OK] 173 individuals | with a 'pre' phase: 169


speed (mm/s)  % still  accel. peaks / min  \
phase strategy                                              
hunt  A                  7.6     34.6                 0.0   
      B                  5.1     26.2                 0.0   
      C                  9.1      9.5                 0.0   
      D                  3.5     45.6                 0.0   
      E                  4.2     46.8                 0.0   
      F                 30.4      7.3                 0.0   
pre   A                  7.1     41.6                 0.0   
      B                  6.7     35.6                 0.0   
      C                 12.2     31.1                 1.0   
      D                  9.2     45.4                 4.0   
      E                  2.6     52.4                 0.0   
      F                 19.5     22.3                 3.0   

                % swimming backwards  turning (°/s)  cuttlefish seen at (°)  \
phase strategy                                                                
hunt  A                          3.0           18.4                   124.0   
      B                          0.0            9.6                   106.0   
      C                          2.9           23.5                   110.4   
      D                          4.3            8.6                    43.5   
      E                          0.7           10.1                    41.3   
      F                          2.2           35.7                    60.4   
pre   A                         11.2           15.4                   108.8   
      B                          6.3           13.1                    74.7   
      C                          4.5           20.7                    88.7   
      D                          7.8           13.1                    65.9   
      E                         10.3            6.1                    56.8   
      F                          4.5           30.8                    99.7   

                distance (mm)  
phase strategy                 
hunt  A                  83.0  
      B                  54.4  
      C                  58.1  
      D                  53.1  
      E                 127.2  
      F                 104.8  
pre   A                 130.9  
      B                 120.4  
      C                 125.7  
      D                 167.3  
      E                 204.8  
      F                 176.2

## 1 — Do the shrimp metrics differ between the cuttlefish strategies?

In [5]:
def _bh(p):
    """Benjamini-Hochberg adjusted p-values."""
    p = np.asarray(p, float)
    o = np.argsort(p)
    q = p[o] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[o] = np.clip(q, 0, 1)
    return out

def kruskal_table(data, metrics, label):
    rows = []
    for m in metrics:
        groups = [data.loc[data["strategy"] == s, m].dropna() for s in STRATS]
        groups = [g for g in groups if len(g) >= 3]
        if len(groups) < 2:
            continue
        H, p = stats.kruskal(*groups)
        n, k = sum(len(g) for g in groups), len(groups)
        rows.append({"phase": label, "metric": m, "H": H, "p": p,
                     "eta²": max(0.0, (H - k + 1) / (n - k)), "n": n,
                     **{f"median {s}": data.loc[data["strategy"] == s, m].median() for s in STRATS}})
    return pd.DataFrame(rows)

tests = pd.concat([kruskal_table(summary[summary["phase"] == ph], METRICS, ph) for ph in ("pre", "hunt")]
                  + [kruskal_table(at_start, list(at_start.columns[:-1]), "at activity start")],
                  ignore_index=True)
tests["p BH"] = _bh(tests["p"])
tests["signif."] = np.select([tests["p BH"] < 0.001, tests["p BH"] < 0.01, tests["p BH"] < 0.05], ["***", "**", "*"], "")
print("Kruskal-Wallis per metric (BH correction over all tests) — eta²: 0.01 small, 0.06 medium, 0.14 large")
tests.round(3)

Kruskal-Wallis per metric (BH correction over all tests) — eta²: 0.01 small, 0.06 medium, 0.14 large


,phase,metric,H,p,eta²,n,median A,median B,median C,median D,median E,median F,p BH,signif.
0,pre,speed (mm/s),28.288,0.000,0.143,169,7.147,6.662,12.190,9.207,2.599,19.486,0.000,***
1,pre,% still,23.852,0.000,0.116,169,41.584,35.624,31.142,45.444,52.444,22.333,0.000,***
2,pre,accel. peaks / min,4.330,0.503,0.000,169,0.000,0.000,1.038,4.000,0.000,3.000,0.571,
3,pre,% swimming backwards,6.563,0.255,0.010,161,11.186,6.267,4.526,7.752,10.303,4.508,0.314,
4,pre,turning (°/s),27.277,0.000,0.137,169,15.364,13.148,20.729,13.139,6.087,30.788,0.000,***
5,pre,cuttlefish seen at (°),31.578,0.000,0.163,169,108.840,74.688,88.742,65.917,56.823,99.671,0.000,***
6,pre,distance (mm),20.844,0.001,0.097,169,130.918,120.390,125.656,167.326,204.826,176.219,0.001,**
7,hunt,speed (mm/s),39.002,0.000,0.204,173,7.608,5.123,9.085,3.487,4.191,30.437,0.000,***
8,hunt,% still,34.917,0.000,0.179,173,34.555,26.225,9.490,45.588,46.791,7.295,0.000,***
9,hunt,accel. peaks / min,4.099,0.535,0.000,173,0.000,0.000,0.000,0.000,0.000,0.000,0.571,


In [6]:
# ── Boxplots: one row per phase, one column per metric ──
def _box(ax, data, m, title):
    vals = [data.loc[data["strategy"] == s, m].dropna().to_numpy() for s in STRATS]
    bp = ax.boxplot(vals, patch_artist=True, widths=0.6, showfliers=False)
    for patch, s in zip(bp["boxes"], STRATS):
        patch.set_facecolor(PAL[s]); patch.set_alpha(0.55)
    for i, v in enumerate(vals):
        ax.scatter(np.random.default_rng(i).normal(i + 1, 0.06, len(v)), v, s=8, color=PAL[STRATS[i]],
                   edgecolor="black", lw=0.3, zorder=3)
    ax.set_xticks(range(1, len(STRATS) + 1)); ax.set_xticklabels(STRATS)
    ax.set_title(title, fontsize=9)
    ax.grid(axis="y", alpha=0.3)

def _sig(phase, m):
    r = tests[(tests["phase"] == phase) & (tests["metric"] == m)]
    return f"\np BH={r['p BH'].iloc[0]:.3f} {r['signif.'].iloc[0]}" if len(r) else ""

for ph in ("pre", "hunt"):
    d = summary[summary["phase"] == ph]
    fig, axes = plt.subplots(1, len(METRICS), figsize=(3.0 * len(METRICS), 3.6))
    for ax, m in zip(axes, METRICS):
        _box(ax, d, m, m + _sig(ph, m))
    fig.suptitle(f"Shrimp behaviour per cuttlefish strategy — phase '{ph}'"
                 + (" (30 s before the cuttlefish activity)" if ph == "pre" else " (cuttlefish activity -> strike)"))
    plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
for ax, m in zip(axes, at_start.columns[:-1]):
    _box(ax, at_start, m, m + _sig("at activity start", m))
fig.suptitle("Shrimp at the start of the cuttlefish activity, per strategy")
plt.tight_layout(); plt.show()

[SAVED] reference_best/figures_5_shrimp_impact/Shrimp_behaviour_per_cuttlefish_strategy_phase_pre_30_s_before_the_cut.png
[SAVED] reference_best/figures_5_shrimp_impact/Shrimp_behaviour_per_cuttlefish_strategy_phase_hunt_cuttlefish_activit.png
[SAVED] reference_best/figures_5_shrimp_impact/Shrimp_at_the_start_of_the_cuttlefish_activity_per_strategy.png


## 2 — Shrimp metrics over time, per cuttlefish strategy (aligned on the start of the cuttlefish activity)

In [7]:
T_MIN, T_MAX, BIN_S = -30, 10, 1.0      # seconds around the start of the cuttlefish activity (0)
TIME_METRICS = {"shrimp_speed_mm_s": "speed (mm/s)", "shrimp_accel_peak": "accel. peaks / s",
                "shrimp_turn_speed_deg_s": "turning (°/s)", "shrimp_view_cuttle": "cuttlefish seen at (°)",
                "distance_mm": "distance (mm)", "shrimp_forward_speed": "forward speed (mm/s, - = backwards)"}

w = sh[(sh["time_from_activity_s"] >= T_MIN) & (sh["time_from_activity_s"] < T_MAX)].copy()
w["bin"] = (np.floor(w["time_from_activity_s"] / BIN_S) * BIN_S + BIN_S / 2).round(3)
w["shrimp_accel_peak"] = w["shrimp_accel_peak"] * 60          # peaks per frame -> per second (60 fps)
per_ind = w.groupby(["strategy", "animal_id", "bin"])[list(TIME_METRICS)].mean().reset_index()

fig, axes = plt.subplots(2, 3, figsize=(16, 7.5), sharex=True)
for ax, (col, lab) in zip(axes.flat, TIME_METRICS.items()):
    for s in STRATS:
        g = per_ind[per_ind["strategy"] == s].groupby("bin")[col]
        m, se, n = g.mean(), g.sem(), g.count()
        m, se = m[n >= 3], se[n >= 3]
        ax.plot(m.index, m, color=PAL[s], lw=1.8, label=s)
        ax.fill_between(m.index, m - se, m + se, color=PAL[s], alpha=0.15)
    ax.axvline(0, color=PHASE_COLOR, ls="--", lw=1.6)
    ax.set_title(lab, fontsize=10); ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("time from the start of the cuttlefish activity (s)")
axes.flat[0].legend(title="cuttlefish\nstrategy", fontsize=8)
fig.suptitle("Shrimp behaviour around the start of the cuttlefish activity (mean ± SEM over individuals; "
             "dashed line = cuttlefish becomes active)")
plt.tight_layout(); plt.show()

[SAVED] reference_best/figures_5_shrimp_impact/Shrimp_behaviour_around_the_start_of_the_cuttlefish_activity_mean_SEM_.png


## 3 — Can the shrimp behaviour BEFORE the cuttlefish activity predict the strategy?

In [8]:
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_val_score, permutation_test_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

PRED_PHASE = "pre"         # "pre" = before the cuttlefish activity (influence) | "hunt" = during (reaction possible)
MIN_PRE_S  = 5             # individuals with less than MIN_PRE_S seconds of "pre" are left out
N_PERM     = 200           # permutations for the p-value

d = summary[summary["phase"] == PRED_PHASE].set_index("animal_id")
if PRED_PHASE == "pre":
    d = d[d["duration (s)"] >= MIN_PRE_S]
d = d.join(at_start.drop(columns="strategy"))
y = d["strategy"].to_numpy()
k = len(np.unique(y))
# Two sets of predictors:
#  - OWN movement of the shrimp only (independent of where the cuttlefish is)
#  - + position relative to the cuttlefish (distance, angle). CAUTION: partly circular, these
#    also describe the cuttlefish, and the SeqVec "start" family uses the start distance
#    to build the strategies
SETS = {"shrimp own movement": ["speed (mm/s)", "% still", "accel. peaks / min", "% swimming backwards", "turning (°/s)"],
        "+ position relative to the cuttlefish": METRICS + list(at_start.columns[:-1])}
print(f"{len(y)} individuals | {k} strategies | chance level (balanced accuracy) = {1 / k:.2f}")

logit = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(max_iter=2000, class_weight="balanced"))
forest = make_pipeline(SimpleImputer(strategy="median"),
                       RandomForestClassifier(n_estimators=300, class_weight="balanced", min_samples_leaf=3, random_state=0))
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
perm_res = {}
for set_name, feats in SETS.items():
    X = d[feats].to_numpy(float)
    print(f"\n── {set_name} ({len(feats)} metrics)")
    for name, model in (("logistic regression", logit), ("random forest", forest)):
        sc = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy")
        print(f"  {name:<20} balanced accuracy = {sc.mean():.2f} ± {sc.std():.2f}")
    score, perm, p = permutation_test_score(logit, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                                            scoring="balanced_accuracy", n_permutations=N_PERM, random_state=0)
    perm_res[set_name] = (score, perm, p)
    print(f"  permutation test (logistic regression): score {score:.2f} | shuffled strategies {perm.mean():.2f} "
          f"± {perm.std():.2f} | p = {p:.3f}  -> {'better than chance' if p < 0.05 else 'NOT better than chance'}")

feat_names = SETS["+ position relative to the cuttlefish"]
X = d[feat_names].to_numpy(float)
forest.fit(X, y)
imp = permutation_importance(forest, X, y, n_repeats=30, random_state=0, scoring="balanced_accuracy")
order = np.argsort(imp.importances_mean)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for (set_name, (score, perm, p)), c in zip(perm_res.items(), ("#44AAFF", "#FF8844")):
    axes[0].hist(perm, bins=25, color=c, alpha=0.35)
    axes[0].axvline(score, color=c, lw=2.5, label=f"{set_name}: {score:.2f} (p = {p:.3f})")
axes[0].axvline(1 / k, color="black", ls=":", label=f"chance {1 / k:.2f}")
axes[0].set_title("Balanced accuracy: real (line) vs shuffled strategies (histogram)", fontsize=10)
axes[0].legend(fontsize=8)
axes[1].barh(np.array(feat_names)[order], imp.importances_mean[order], xerr=imp.importances_std[order], color="#44AAFF")
axes[1].set_title("Which metric helps the prediction? (all metrics, permutation importance, random forest)", fontsize=10)
plt.tight_layout(); plt.show()

163 individuals | 6 strategies | chance level (balanced accuracy) = 0.17

── shrimp own movement (5 metrics)
  logistic regression  balanced accuracy = 0.23 ± 0.07
  random forest        balanced accuracy = 0.19 ± 0.07
  permutation test (logistic regression): score 0.23 | shuffled strategies 0.17 ± 0.04 | p = 0.065  -> NOT better than chance

── + position relative to the cuttlefish (9 metrics)
  logistic regression  balanced accuracy = 0.45 ± 0.08
  random forest        balanced accuracy = 0.50 ± 0.07
  permutation test (logistic regression): score 0.46 | shuffled strategies 0.17 ± 0.04 | p = 0.005  -> better than chance
[SAVED] reference_best/figures_5_shrimp_impact/Balanced_accuracy_real_line_vs_shuffled_strategies_histogram.png


## 4 — Which shrimp strategy faces which cuttlefish strategy? (correspondence matrix)
Shrimp groups from `SHRIMP_SESSION` (1_compute with `SUBJECT = "shrimp"`); if `SHRIMP_SESSION = None`,
the previous shrimp strategies `Data/mapping_optimal_shrimp.json` are used (as at the end of all_cuttlefish).

In [9]:
# ══════════════════════════════════════════════════════════════
#  CORRESPONDENCE MATRIX — Cuttlefish × Shrimp strategies
# ══════════════════════════════════════════════════════════════
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.metrics import adjusted_rand_score

SHRIMP_MAPPING_FILE = DATA_DIR / "mapping_optimal_shrimp.json"   # used when SHRIMP_SESSION = None
EXCLUDE_MONO        = True                                        # leave out the MONO groups

# ── Mappings {id: letter} ──
map_cuttle = {str(k): v for k, v in mapping_cuttle.items()}
if mapping_shrimp is not None:
    map_shrimp, shrimp_src = {str(k): v for k, v in mapping_shrimp.items()}, SHRIMP_SESSION
else:
    map_shrimp, shrimp_src = ({str(k): v for k, v in json.load(open(SHRIMP_MAPPING_FILE)).items()},
                              SHRIMP_MAPPING_FILE.name)

common_ids = set(map_cuttle) & set(map_shrimp)
print(f"[OK] {len(common_ids)} common individuals")
print(f"     Cuttlefish ({CUTTLE_SESSION}) : {len(map_cuttle)} individuals | groups = {sorted(set(map_cuttle.values()))}")
print(f"     Shrimp     ({shrimp_src}) : {len(map_shrimp)} individuals | groups = {sorted(set(map_shrimp.values()))}")
if not common_ids:
    raise ValueError("No common individual — check that the animal_id values are identical")

df_match = pd.DataFrame([{"animal_id": aid, "strat_cuttle": map_cuttle[aid], "strat_shrimp": map_shrimp[aid]}
                         for aid in sorted(common_ids, key=int)])
if EXCLUDE_MONO:
    df_match = df_match[(df_match["strat_cuttle"] != "MONO") & (df_match["strat_shrimp"] != "MONO")]
print(f"[OK] {len(df_match)} individuals" + (" (MONO excluded)" if EXCLUDE_MONO else ""))

# ── Contingency table ──
ct_count = pd.crosstab(df_match["strat_cuttle"], df_match["strat_shrimp"])
ct_count.index.name, ct_count.columns.name = "Cuttlefish strategy", "Shrimp strategy"
ct_row = ct_count.div(ct_count.sum(axis=1), axis=0)            # per cuttlefish strategy (rows = 100 %)
ct_col = ct_count.div(ct_count.sum(axis=0), axis=1)            # per shrimp strategy (columns = 100 %)

chi2, p, dof, expected = chi2_contingency(ct_count)
cramer_v = np.sqrt(chi2 / (ct_count.values.sum() * (min(ct_count.shape) - 1)))
ari      = adjusted_rand_score(df_match["strat_cuttle"], df_match["strat_shrimp"])
sig      = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
stat_txt = f"χ²={chi2:.1f}  df={dof}  p={p:.4f} ({sig}) | Cramér's V={cramer_v:.2f} | ARI={ari:.2f}"
print(f"\n[CHI²] {stat_txt}")
if (expected < 5).mean() > 0.2:
    print("       ⚠️  more than 20 % of the expected counts < 5: interpret the p-value with care")
df_resid = pd.DataFrame((ct_count.values - expected) / np.sqrt(expected), index=ct_count.index, columns=ct_count.columns)

src_txt = f"cuttlefish: {CUTTLE_SESSION} | shrimp: {shrimp_src}"

def _heat(data, annot, fmt, cmap, title, cbar, **kw):
    fig, ax = plt.subplots(figsize=(max(6, len(data.columns) * 1.2), max(4, len(data) * 0.8)))
    fig.patch.set_facecolor("#1a1a2e")
    sns.heatmap(data, annot=annot, fmt=fmt, cmap=cmap, ax=ax, linewidths=0.5, linecolor="#333333",
                cbar_kws={"label": cbar}, **kw)
    ax.set_title(title, color="#cccccc", fontsize=10)
    ax.set_xlabel("Shrimp strategy", color="#aaaaaa"); ax.set_ylabel("Cuttlefish strategy", color="#aaaaaa")
    ax.tick_params(colors="#aaaaaa")
    plt.tight_layout(); plt.show()

# ── FIGURE 1 — raw counts ──
_heat(ct_count, True, "d", "YlOrRd", f"Cuttlefish × Shrimp correspondence — raw counts\n{stat_txt}\n{src_txt}",
      "Number of individuals")

# ── FIGURE 2 — for each cuttlefish strategy, which shrimp strategy? (rows = 100 %) ──
annot_row = ct_count.astype(str) + "\n(" + (ct_row * 100).round(0).astype(int).astype(str) + "%)"
_heat(ct_row, annot_row, "", "Blues", "For each CUTTLEFISH strategy → distribution of SHRIMP strategies\n"
      "(each row sums to 100%)", "Proportion (per row = per cuttlefish)", vmin=0, vmax=1)

# ── FIGURE 3 — for each shrimp strategy, which cuttlefish strategy? (columns = 100 %) ──
annot_col = ct_count.astype(str) + "\n(" + (ct_col * 100).round(0).astype(int).astype(str) + "%)"
_heat(ct_col, annot_col, "", "Greens", "For each SHRIMP strategy → distribution of CUTTLEFISH strategies\n"
      "(each column sums to 100%)", "Proportion (per column = per shrimp)", vmin=0, vmax=1)

# ── FIGURE 4 — standardised residuals ──
_heat(df_resid, True, ".1f", "RdBu_r", f"Standardised residuals — Cuttlefish × Shrimp\n{stat_txt}\n"
      "> +2 = over-represented association  |  < -2 = under-represented", "Standardised residual",
      center=0, vmin=-4, vmax=4)

# ── FIGURE 5 — stacked bars, both directions ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor("#1a1a2e")
sh_pal = {s: plt.cm.tab10(i % 10) for i, s in enumerate(ct_count.columns)}
for ax, data, pal, xl, leg in ((axes[0], ct_row, sh_pal, "Cuttlefish strategy", "Shrimp strategy"),
                               (axes[1], ct_col.T, {s: PAL.get(s, "#888888") for s in ct_count.index},
                                "Shrimp strategy", "Cuttlefish strategy")):
    bottom = np.zeros(len(data))
    for c in data.columns:
        ax.bar(data.index, data[c], bottom=bottom, color=pal[c], edgecolor="#1a1a2e", label=c)
        for i, (v, b) in enumerate(zip(data[c], bottom)):
            if v >= 0.08:
                ax.text(i, b + v / 2, f"{v * 100:.0f}%", ha="center", va="center", fontsize=8, color="black")
        bottom += data[c].to_numpy()
    ax.set_facecolor("#0d0d1a"); ax.tick_params(colors="#aaaaaa")
    ax.set_xlabel(xl, color="#aaaaaa"); ax.set_ylabel("proportion", color="#aaaaaa"); ax.set_ylim(0, 1)
    ax.legend(title=leg, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
axes[0].set_title("Shrimp strategy faced by each cuttlefish strategy", color="#cccccc")
axes[1].set_title("Cuttlefish strategy used against each shrimp strategy", color="#cccccc")
fig.suptitle(f"{stat_txt}\n{src_txt}", color="#cccccc", fontsize=10)
plt.tight_layout(); plt.show()

# ── Text summary ──
print(f"\n{'=' * 55}\nNOTABLE ASSOCIATIONS (residual > +2 or < -2)\n{'=' * 55}")
for i in df_resid.index:
    for j in df_resid.columns:
        r = df_resid.loc[i, j]
        if abs(r) >= 2:
            print(f"  Cuttlefish {i} × Shrimp {j} : residual={r:.1f}  n={ct_count.loc[i, j]}  → "
                  f"{'OVER' if r > 0 else 'UNDER'}-represented")

[OK] 183 common individuals
     Cuttlefish (reference_best) : 183 individuals | groups = ['A', 'B', 'C', 'D', 'E', 'F']
     Shrimp     (mapping_optimal_shrimp.json) : 202 individuals | groups = ['A', 'B', 'C', 'D', 'E', 'F', 'MONO']
[OK] 167 individuals (MONO excluded)

[CHI²] χ²=126.1  df=25  p=0.0000 (***) | Cramér's V=0.39 | ARI=0.15
       ⚠️  more than 20 % of the expected counts < 5: interpret the p-value with care
[SAVED] reference_best/figures_5_shrimp_impact/Cuttlefish_Shrimp_correspondence_raw_counts_χ²_126_1_df_25_p_0_0000_Cr.png
[SAVED] reference_best/figures_5_shrimp_impact/For_each_CUTTLEFISH_strategy_distribution_of_SHRIMP_strategies_each_ro.png
[SAVED] reference_best/figures_5_shrimp_impact/For_each_SHRIMP_strategy_distribution_of_CUTTLEFISH_strategies_each_co.png
[SAVED] reference_best/figures_5_shrimp_impact/Standardised_residuals_Cuttlefish_Shrimp_χ²_126_1_df_25_p_0_0000_Cramé.png
[SAVED] reference_best/figures_5_shrimp_impact/χ²_126_1_df_25_p_0_0000_Cramér_s_V_0_3

## 5 — Speed correlation: does the cuttlefish change speed before or after the shrimp?
Cuttlefish speed and shrimp speed on the same graph (+ distance), per individual and per strategy,
on the **hunting phase** by default (`XCORR_PHASE`); the per-strategy curves use a normalised hunting time
(0 = cuttlefish becomes active, 1 = strike) because the hunts last from 1 to 60 s.
Two measures of the delay:
- **cross-correlation**: the lag that best aligns the two speed curves;
- **latency between peaks**: for each shrimp speed peak, the nearest cuttlefish speed peak.

Sign convention: **lag > 0 = the cuttlefish changes speed AFTER the shrimp** (it reacts to the shrimp);
**lag < 0 = the cuttlefish changes BEFORE** (the shrimp reacts to the cuttlefish).
Figures: `Results/<CUTTLE_SESSION>/figures_5_shrimp_impact/speed_correlation/` (one per individual in `individuals/`).

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SPEED CORRELATION — computation (per individual)
# ══════════════════════════════════════════════════════════════
from scipy.signal import find_peaks
from matplotlib.figure import Figure

XCORR_PHASE     = "hunt"   # part of the window used (delays AND figures): "hunt" (cuttlefish activity -> strike)
                           #   | "all" (30 s before + hunt) | "pre"
LAG_MAX_S       = 1.0      # delays explored: ± LAG_MAX_S seconds (hunts last 1.8 s for 10 % of the individuals,
                           #   4.8 s median: keep it well below the hunt duration)
PEAK_MIN_MM_S   = 30.0     # a speed peak must reach at least this speed (mm/s)
PEAK_PROM_MM_S  = 15.0     # ... and stand out from its surroundings by at least this much (prominence, mm/s)
PEAK_GAP_S      = 0.5      # two peaks of the same animal at least this far apart (s)
INDIVIDUAL_FIGS = True     # one figure per individual (saved in individuals/, not displayed)
FPS_SC          = 60
C_CUTTLE, C_SHRIMP, C_DIST = "#1f77b4", "#ff7f0e", "#777777"

SPEED_DIR = LOAD_DIR / "figures_5_shrimp_impact" / "speed_correlation"
(SPEED_DIR / "individuals").mkdir(parents=True, exist_ok=True)

_sc_cols = ["animal_id", "frame", "phase", "time_from_activity_s", "shrimp_speed_mm_s", "cuttle_speed_mm_s", "distance_mm"]
_head = pd.read_csv(SHRIMP_CSV, nrows=1).columns
if "cuttle_speed_mm_s" not in _head:
    raise RuntimeError("cuttle_speed_mm_s missing from the shrimp CSV: rerun Code/add_features_shrimp.py")
sc = pd.read_csv(SHRIMP_CSV, usecols=_sc_cols)
sc = sc[sc["animal_id"].isin(list(mapping_cuttle))].sort_values(["animal_id", "frame"])
sc["strategy"] = sc["animal_id"].map(mapping_cuttle)
max_lag = int(LAG_MAX_S * FPS_SC)


def _xcorr(a, b, max_lag):
    """r(L) = Pearson correlation between a(t) and b(t + L), L = -max_lag..max_lag frames.
    a = shrimp, b = cuttlefish -> the best L > 0 means the cuttlefish follows the shrimp."""
    lags = np.arange(-max_lag, max_lag + 1)
    r = np.full(len(lags), np.nan)
    for i, L in enumerate(lags):
        x, y = (a[:len(a) - L], b[L:]) if L >= 0 else (a[-L:], b[:len(b) + L])
        if len(x) > 30 and x.std() > 0 and y.std() > 0:
            r[i] = np.corrcoef(x, y)[0, 1]
    return lags, r


def _peaks(v):
    p, _ = find_peaks(v, height=PEAK_MIN_MM_S, prominence=PEAK_PROM_MM_S, distance=int(PEAK_GAP_S * FPS_SC))
    return p


def _latencies(ps, pc):
    """For each shrimp peak: delay (s) to the nearest cuttlefish peak within ± LAG_MAX_S."""
    out = []
    for p in ps:
        if len(pc):
            d = pc - p
            j = np.argmin(np.abs(d))
            if abs(d[j]) <= max_lag:
                out.append(d[j] / FPS_SC)
    return out


rows, lat_rows, xc_curves = [], [], {}
for aid, g in sc.groupby("animal_id"):
    strat = g["strategy"].iloc[0]
    w = g if XCORR_PHASE == "all" else g[g["phase"] == XCORR_PHASE]
    a, b = w["shrimp_speed_mm_s"].to_numpy(float), w["cuttle_speed_mm_s"].to_numpy(float)
    lags, r = _xcorr(a, b, max_lag)
    ok = np.isfinite(r).any()
    best = int(np.nanargmax(r)) if ok else None
    ps, pc = _peaks(a), _peaks(b)
    lat = _latencies(ps, pc)
    lat_rows += [{"animal_id": aid, "strategy": strat, "latency (s)": x} for x in lat]
    if ok:
        xc_curves[aid] = r
    rows.append({"animal_id": aid, "strategy": strat,
                 "best lag (s)": lags[best] / FPS_SC if ok else np.nan,
                 "r at best lag": r[best] if ok else np.nan,
                 "r at lag 0": r[max_lag] if ok else np.nan,
                 "shrimp peaks": len(ps), "cuttlefish peaks": len(pc),
                 "matched peaks": len(lat), "median peak latency (s)": np.median(lat) if lat else np.nan})

    if INDIVIDUAL_FIGS:
        t = w["time_from_activity_s"].to_numpy(float)
        sv, cv, dv = (w[c].to_numpy(float) for c in ("shrimp_speed_mm_s", "cuttle_speed_mm_s", "distance_mm"))
        psg, pcg = _peaks(sv), _peaks(cv)
        fig = Figure(figsize=(13, 6.5))
        ax, axc = fig.subplots(2, 1, gridspec_kw=dict(height_ratios=[3, 1.3]))
        ax.plot(t, sv, color=C_SHRIMP, lw=1, label="shrimp speed")
        ax.plot(t, cv, color=C_CUTTLE, lw=1, label="cuttlefish speed")
        ax.plot(t[psg], sv[psg], "v", color=C_SHRIMP, ms=6, mec="black", mew=0.5, label="shrimp peaks")
        ax.plot(t[pcg], cv[pcg], "^", color=C_CUTTLE, ms=6, mec="black", mew=0.5, label="cuttlefish peaks")
        ax.axvline(0, color=PHASE_COLOR, ls="--", lw=1.5, label="cuttlefish becomes active")
        ax.set_ylabel("speed (mm/s)"); ax.set_xlabel("time from the start of the cuttlefish activity (s)")
        axd = ax.twinx()
        axd.plot(t, dv, color=C_DIST, ls="--", lw=1.3, label="distance")
        axd.set_ylabel("cuttlefish–shrimp distance (mm)", color=C_DIST)
        h1, l1 = ax.get_legend_handles_labels(); h2, l2 = axd.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left", ncol=3)
        ax.grid(alpha=0.3)
        if ok:
            axc.plot(lags / FPS_SC, r, color="black", lw=1.2)
            axc.axvline(0, color="#999999", lw=0.8)
            axc.axvline(lags[best] / FPS_SC, color="red", ls="--", lw=1.2,
                        label=f"best lag {lags[best] / FPS_SC:+.2f} s (r = {r[best]:.2f})")
            axc.legend(fontsize=7, loc="upper right")
        axc.set_xlabel("lag (s)   > 0 = cuttlefish AFTER the shrimp | < 0 = cuttlefish BEFORE")
        axc.set_ylabel("correlation"); axc.grid(alpha=0.3)
        fig.suptitle(f"ID {aid} — strategy {strat} | cross-correlation on '{XCORR_PHASE}' | "
                     f"{len(lat)} matched peaks" + (f", median latency {np.median(lat):+.2f} s" if lat else ""))
        fig.tight_layout()
        fig.savefig(SPEED_DIR / "individuals" / f"{strat}_ID{aid}.png", dpi=100)

speed_corr = pd.DataFrame(rows)
peak_lat   = pd.DataFrame(lat_rows)
speed_corr.to_csv(SPEED_DIR / "speed_correlation_individuals.csv", index=False)
print(f"[OK] {len(speed_corr)} individuals | {len(peak_lat)} matched peak pairs"
      + (f" | {len(speed_corr)} figures → {SPEED_DIR / 'individuals'}" if INDIVIDUAL_FIGS else ""))
speed_corr.groupby("strategy")[["best lag (s)", "r at best lag", "r at lag 0", "matched peaks",
                                "median peak latency (s)"]].median().round(2)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SPEED CORRELATION — per strategy (figures + tests)
# ══════════════════════════════════════════════════════════════
SC_N_BINS = 40      # hunting phase cut into SC_N_BINS bins of NORMALISED time (0 = cuttlefish becomes active, 1 = strike)

# ── Summary + tests: is the lag different from 0? different between strategies? ──
summ = []
for s in STRATS:
    v = speed_corr.loc[speed_corr["strategy"] == s, "best lag (s)"].dropna()
    l = peak_lat.loc[peak_lat["strategy"] == s, "latency (s)"] if len(peak_lat) else pd.Series(dtype=float)
    summ.append({"strategy": s, "n": len(v), "median best lag (s)": v.median(),
                 "% cuttlefish AFTER shrimp": (v > 0).mean() * 100,
                 "Wilcoxon p (lag ≠ 0)": stats.wilcoxon(v).pvalue if len(v) >= 6 and (v != 0).any() else np.nan,
                 "peak pairs": len(l), "median peak latency (s)": l.median() if len(l) else np.nan})
summ = pd.DataFrame(summ)
_g = [speed_corr.loc[speed_corr["strategy"] == s, "best lag (s)"].dropna() for s in STRATS]
_kw = stats.kruskal(*[x for x in _g if len(x) >= 3])
print(f"Best lag between strategies: Kruskal-Wallis H = {_kw.statistic:.1f}, p = {_kw.pvalue:.4f}")
print(summ.round(3).to_string(index=False))
summ.to_csv(SPEED_DIR / "speed_correlation_per_strategy.csv", index=False)

# ── FIGURE 1 — mean speeds (cuttlefish + shrimp) and distance over time, one panel per strategy ──
# Each hunt lasts 1 to 60 s: in seconds, the individuals would drop out one after the other and the strike
# would not be at the same place -> the hunt of EVERY individual is rescaled to 0..1, then averaged
w = sc[sc["phase"] == "hunt"].copy()
_t = w.groupby("animal_id")["time_from_activity_s"]
w["t_norm"] = (w["time_from_activity_s"] - _t.transform("min")) / (_t.transform("max") - _t.transform("min")).replace(0, np.nan)
w["bin"] = (np.minimum(np.floor(w["t_norm"] * SC_N_BINS), SC_N_BINS - 1) + 0.5) / SC_N_BINS
pi = w.dropna(subset=["bin"]).groupby(["strategy", "animal_id", "bin"])[["shrimp_speed_mm_s", "cuttle_speed_mm_s", "distance_mm"]].mean().reset_index()
nc = min(3, len(STRATS)); nr = int(np.ceil(len(STRATS) / nc))
fig, axes = plt.subplots(nr, nc, figsize=(6 * nc, 4 * nr), sharex=True, squeeze=False)
for ax, s in zip(axes.flat, STRATS):
    d = pi[pi["strategy"] == s].groupby("bin")
    n = d["animal_id"].nunique()
    axd = ax.twinx()
    for col, c, lab, a_ in (("shrimp_speed_mm_s", C_SHRIMP, "shrimp speed", ax),
                            ("cuttle_speed_mm_s", C_CUTTLE, "cuttlefish speed", ax),
                            ("distance_mm", C_DIST, "distance", axd)):
        m, se = d[col].mean()[n >= 3], d[col].sem()[n >= 3]
        a_.plot(m.index, m, color=c, lw=1.8, ls="--" if col == "distance_mm" else "-", label=lab)
        a_.fill_between(m.index, m - se, m + se, color=c, alpha=0.15)
    ax.set_title(f"Strategy {s} (n={speed_corr['strategy'].eq(s).sum()})", color=PAL[s], fontweight="bold")
    ax.set_ylabel("speed (mm/s)"); axd.set_ylabel("distance (mm)", color=C_DIST)
    ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("normalised hunting time (0 = cuttlefish becomes active, 1 = strike)")
for ax in list(axes.flat)[len(STRATS):]:
    ax.set_visible(False)
h1, l1 = axes.flat[0].get_legend_handles_labels()
fig.legend(h1 + [plt.Line2D([], [], color=C_DIST, ls="--")], l1 + ["distance"], loc="upper right", ncol=3)
fig.suptitle("Cuttlefish and shrimp speed + distance during the HUNT, per strategy\n"
             "(each hunt rescaled to 0–1, then mean ± SEM over individuals)", y=1.02)
plt.tight_layout()
fig.savefig(SPEED_DIR / "speeds_and_distance_per_strategy_hunt.png", dpi=120, bbox_inches="tight")
plt.show()

# ── FIGURE 2 — cross-correlation + delays per strategy ──
lag_s = np.arange(-max_lag, max_lag + 1) / FPS_SC
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for s in STRATS:
    ids = [a for a in speed_corr.loc[speed_corr["strategy"] == s, "animal_id"] if a in xc_curves]
    if not ids:
        continue
    M = np.vstack([xc_curves[a] for a in ids])
    m, se = np.nanmean(M, 0), np.nanstd(M, 0) / np.sqrt(np.sum(np.isfinite(M), 0))
    axes[0].plot(lag_s, m, color=PAL[s], lw=2, label=f"{s} (n={len(ids)})")
    axes[0].fill_between(lag_s, m - se, m + se, color=PAL[s], alpha=0.12)
axes[0].axvline(0, color="#999999", lw=0.8)
axes[0].set_xlabel("lag (s)   > 0 = cuttlefish AFTER the shrimp"); axes[0].set_ylabel("mean correlation")
axes[0].set_title("Mean cross-correlation shrimp → cuttlefish speed"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

for ax, data, col, title in ((axes[1], speed_corr, "best lag (s)", f"Best lag per individual (KW p = {_kw.pvalue:.3f})"),
                             (axes[2], peak_lat, "latency (s)", "Latency shrimp peak → nearest cuttlefish peak")):
    vals = [data.loc[data["strategy"] == s, col].dropna().to_numpy() for s in STRATS] if len(data) else [[] for _ in STRATS]
    bp = ax.boxplot(vals, patch_artist=True, widths=0.6, showfliers=False)
    for patch, s in zip(bp["boxes"], STRATS):
        patch.set_facecolor(PAL[s]); patch.set_alpha(0.55)
    for i, v in enumerate(vals):
        ax.scatter(np.random.default_rng(i).normal(i + 1, 0.06, len(v)), v, s=8, color=PAL[STRATS[i]],
                   edgecolor="black", lw=0.3, zorder=3)
    ax.axhline(0, color="#999999", lw=0.8)
    ax.set_xticks(range(1, len(STRATS) + 1)); ax.set_xticklabels([f"{s}\nn={len(v)}" for s, v in zip(STRATS, vals)])
    ax.set_ylabel(f"{col}   > 0 = cuttlefish AFTER"); ax.set_title(title); ax.grid(axis="y", alpha=0.3)
fig.suptitle(f"Who changes speed first? (cross-correlation on '{XCORR_PHASE}', lags ± {LAG_MAX_S:.0f} s)")
plt.tight_layout()
fig.savefig(SPEED_DIR / "lags_per_strategy.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"[OK] figures + tables → {SPEED_DIR}")